# Pipeline, особенности моделей и преобразования таргета

In [26]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display='diagram')  # красивые HTML-диаграммы для Pipeline
rng = np.random.default_rng(42)


---
# Часть 1. sklearn Pipeline

В первой части мы разберём, как правильно соединять шаги предобработки данных и обучения модели в единый воспроизводимый конвейер.

## 1.1 Простой Pipeline

`Pipeline` — это объект sklearn, который последовательно выполняет список шагов. Последний шаг должен быть моделью, все предыдущие — трансформерами.

Есть два способа создать Pipeline:
- `Pipeline([('step1', obj1), ('step2', obj2)])` — явные имена
- `make_pipeline(obj1, obj2)` — имена генерируются автоматически из класса

Sklearn умеет красиво отображать Pipeline в HTML-виде прямо в ноутбуке — мы уже включили это настройкой `set_config(display='diagram')` в первой ячейке.

In [27]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline, Pipeline

# Вариант 1: make_pipeline (имена генерируются автоматически)
pipe_auto = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
pipe_auto  # <-- в ноутбуке рендерится как интерактивная диаграмма


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None


In [29]:
# Вариант 2: явные имена шагов через Pipeline
from sklearn.preprocessing import MinMaxScaler


pipe_named = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model',  Ridge(alpha=1.0)),
])
pipe_named


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None


In [ ]:
# Доступ к шагам через .named_steps или через атрибут
pipe_named.fit([[1], [2], [3]], [1, 2, 3])

print('Среднее после обучения scaler:', pipe_named.named_steps['scaler'].mean_)
print('Коэффициент Ridge:', pipe_named.named_steps['model'].coef_)


### Как Pipeline ведёт себя при `fit` и `predict`

| Метод | Что происходит |
|-------|----------------|
| `pipe.fit(X_train, y_train)` | Для каждого трансформера вызывается `fit_transform`, для модели — `fit` |
| `pipe.predict(X_test)` | Для каждого трансформера вызывается `transform`, для модели — `predict` |
| `pipe.score(X_test, y_test)` | То же, что и `predict`, плюс вычисление метрики |

Т.е. на тесте трансформеры не переобучаются — только применяют параметры, выученные на трейне.

## 1.2 ColumnTransformer: разные шаги для разных признаков

В реальных задачах признаки бывают разных типов: числовые, категориальные, текстовые. К ним нужно применять разные трансформации. `ColumnTransformer` позволяет указать, **какой трансформер применять к каким столбцам**, и упаковать это в один объект.

### Синтетические данные о зарплатах

Создадим датасет с такими признаками:
- `age` — возраст (число)
- `experience` — стаж (число)
- `city` — город (категория)
- `education` — уровень образования (категория)
- `salary` — зарплата (таргет)

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 500

cities = ['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск', 'Екатеринбург']
educations = ['среднее', 'бакалавр', 'магистр', 'PhD']
city_bonus = {
    'Москва': 30_000,
    'Санкт-Петербург': 20_000,
    'Казань': 5_000,
    'Новосибирск': 7_000,
    'Екатеринбург': 8_000,
}
edu_bonus = {
    'среднее': 0,
    'бакалавр': 15_000,
    'магистр': 25_000,
    'PhD': 40_000,
}

age = rng.integers(22, 60, n)
experience = np.clip(age - 22 + rng.integers(-3, 4, n), 0, 40).astype(float)
city = rng.choice(cities, n)
education = rng.choice(educations, n)

salary = (
        40_000
        + 1_200 * experience
        + np.array([city_bonus[c] for c in city])
        + np.array([edu_bonus[e] for e in education])
        + rng.normal(0, 8_000, n)
)

salary = (salary / 100).round() * 100

df = pd.DataFrame(
    {
        'age': age,
        'experience': experience,
        'city': city,
        'education': education,
        'salary': salary,
    }
)

df.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

num_features = ['age', 'experience']
cat_features = ['city', 'education']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features),
    ],
    remainder='drop',   # остальные колонки не включаем
)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge()),
])

full_pipeline  # диаграмма


In [ ]:
from sklearn.model_selection import cross_validate

X_df = df.drop(columns='salary')
y_df = df['salary']

cv_res = cross_validate(
    full_pipeline, X_df, y_df,
    cv=5, scoring='r2',
    return_train_score=True,
    n_jobs=-1,
)

pd.DataFrame(cv_res).round(3)


Обратите внимание: `cross_validate` вызывается прямо на сыром `DataFrame` — Pipeline сам разберётся, какой трансформер применить к каким колонкам. На каждом фолде `preprocessor` обучается только на тренировочных данных, и никакой утечки нет.

> **Совет:** `set_output(transform='pandas')` позволяет трансформерам возвращать DataFrame вместо numpy-массива — полезно для отладки.

In [ ]:
# Посмотрим, как выглядят данные после preprocessor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_df, test_size=0.2, random_state=0
)

preprocessor.set_output(transform='pandas')
X_train_transformed = preprocessor.fit_transform(X_train)
print('Форма до:', X_train.shape)
print('Форма после:', X_train_transformed.shape)
X_train_transformed.head(3)


In [ ]:
X_test_transformed = preprocessor.transform(X_test)
print('Форма до:', X_test.shape)
print('Форма после:', X_test_transformed.shape)
X_test_transformed.head(3)

## 1.3 Pipeline + GridSearchCV

Pipeline отлично сочетается с `GridSearchCV`. Чтобы обратиться к параметру конкретного шага, используется синтаксис:

```
имя_шага__имя_параметра
```

Например, `model__alpha` — параметр `alpha` шага `model`. А `preprocessor__cat__drop` — параметр `drop` для `cat`-трансформера внутри `preprocessor`.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'preprocessor__cat__drop': ['first', None],
}

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True,
)

grid_search.fit(X_df, y_df)

print('Лучшие параметры:', grid_search.best_params_)
print(f'Лучший R² (CV): {grid_search.best_score_:.4f}')


In [ ]:
# Можно посмотреть всю таблицу результатов
results_df = pd.DataFrame(grid_search.cv_results_)
cols = ['param_model__alpha', 'param_preprocessor__cat__drop',
        'mean_train_score', 'mean_test_score', 'std_test_score']
results_df[cols].sort_values('mean_test_score', ascending=False).round(3)


### Итог по Pipeline

| Что | Почему важно |
|-----|--------------|
| `Pipeline` | Устраняет утечку данных, делает код компактным |
| `ColumnTransformer` | Разные трансформации для числовых и категориальных признаков |
| `GridSearchCV` + Pipeline | Поиск гиперпараметров без дополнительного кода |
| `set_config(display='diagram')` | Удобная визуализация структуры пайплайна |


---
# Часть 2. Особенности моделей

В этой части мы «поиграем» с пятью моделями регрессии и посмотрим, в чём каждая из них хороша и в чём ограничена:

- `LinearRegression` — линейная регрессия
- `DecisionTreeRegressor` — дерево решений
- `RandomForestRegressor` — случайный лес
- `HistGradientBoostingRegressor` — градиентный бустинг
- `KNeighborsRegressor` — метод k ближайших соседей


## 2.1 Как модели аппроксимируют одномерную кривую

Сначала разберём самый простой случай: одна числовая переменная. Сгенерируем данные $y = \sin(x) + \varepsilon$ на отрезке $[0, 2\pi]$ и обучим все пять моделей.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

rng_m = np.random.default_rng(0)

n_train = 80
X_sin = np.sort(rng_m.uniform(0, 2 * np.pi, n_train)).reshape(-1, 1)
y_sin = np.sin(X_sin.ravel()) + rng_m.normal(0, 0.2, n_train)

# Плотная сетка для предсказаний
X_grid = np.linspace(0, 2 * np.pi, 500).reshape(-1, 1)

models = {
    'LinearRegression':             LinearRegression(),
    'DecisionTree (depth=5)':        DecisionTreeRegressor(max_depth=5, random_state=0),
    'RandomForest':                  RandomForestRegressor(n_estimators=100, random_state=0),
    'GradientBoosting':              HistGradientBoostingRegressor(random_state=0),
    'KNN (k=5)':                     KNeighborsRegressor(n_neighbors=5),
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True, constrained_layout=True)

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_sin, y_sin)
    y_pred = model.predict(X_grid)
    ax.scatter(X_sin, y_sin, s=15, alpha=0.6, color='steelblue', label='данные')
    ax.plot(X_grid, np.sin(X_grid), color='gray', lw=1, linestyle='--', label='истина')
    ax.plot(X_grid, y_pred, color='crimson', lw=2, label='модель')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('x')

axes[0].set_ylabel('y')
axes[0].legend(fontsize=8)
fig.suptitle('Аппроксимация sin(x) разными моделями', fontsize=13)


Что мы видим:

| Модель | Характер кривой |
|--------|-----------------|
| **LinearRegression** | Прямая линия — не может воспроизвести нелинейность |
| **DecisionTree** | Ступенчатая кривая — кусочно-константная аппроксимация |
| **RandomForest** | Сглаженная ступенчатость — много деревьев, но форма та же |
| **GradientBoosting** | Почти гладкая кривая — ближе всего к истине |
| **KNN** | Ломаная — предсказание = среднее $k$ соседей, шумит на краях |


## 2.2 Деревья vs линейные модели: что кому трудно

Обе семьи моделей хороши — просто в разных ситуациях. Посмотрим два контрастных примера:

1. **Линейная зависимость** $y = 2x + 3 + \varepsilon$: здесь линейная регрессия должна быть идеальна.
2. **Нелинейная/пилообразная зависимость** $y = |\sin(2x)| + \varepsilon$: здесь линейная модель бессильна.

Для каждого примера сравним `LinearRegression` и `DecisionTree`.

In [ ]:
rng2 = np.random.default_rng(7)
n2 = 100
X2 = np.sort(rng2.uniform(0, 5, n2)).reshape(-1, 1)

# Задача 1: линейная зависимость
y_linear = 2 * X2.ravel() + 3 + rng2.normal(0, 0.5, n2)

# Задача 2: пилообразная нелинейная зависимость
y_nonlin = np.abs(np.sin(2 * X2.ravel())) + rng2.normal(0, 0.1, n2)

X2_grid = np.linspace(0, 5, 400).reshape(-1, 1)

lr  = LinearRegression()
dt  = DecisionTreeRegressor(max_depth=6, random_state=0)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

datasets = [
    (y_linear, 'y = 2x + 3 + шум (линейная)'),
    (y_nonlin, r'y = |sin(2x)| + шум (нелинейная)'),
]

for row, (y_data, title) in enumerate(datasets):
    for col, (model, mname) in enumerate([(lr, 'LinearRegression'), (dt, 'DecisionTree')]):
        ax = axes[row][col]
        model.fit(X2, y_data)
        y_hat = model.predict(X2_grid)
        ax.scatter(X2, y_data, s=15, alpha=0.5, color='steelblue')
        ax.plot(X2_grid, y_hat, color='crimson', lw=2)
        ax.set_title(f'{mname}\n{title}', fontsize=9)
        ax.set_xlabel('x')
        ax.set_ylabel('y')

fig.suptitle('Линейные модели vs деревья на разных задачах', fontsize=12)


**Выводы:**

- Линейная регрессия идеально справляется с линейной зависимостью и совершенно не может воспроизвести пилообразную.
- Дерево аппроксимирует обе зависимости через ступеньки — для линейной это выглядит «грубо», но в целом верно. Для нелинейной — очень хорошо.

> **Практический совет.** Если у вас есть основание думать, что зависимость линейная (или вы хотите интерпретируемую модель) — берите линейную регрессию. Если зависимость сложная или неизвестна — деревья/бустинг работают из коробки без дополнительной инженерии признаков.

## 2.3 Инвариантность деревьев к масштабированию

Дерево решений разбивает пространство признаков на прямоугольные области по правилам вида $x_j < t$. Если умножить признак на константу, порог пропорционально изменится — но структура дерева останется той же.

Поэтому **деревьям нормировка не нужна**. А вот `KNeighborsRegressor` и `LinearRegression` ведут себя совсем иначе без нормировки.

Сначала посмотрим, как выглядят признаки до и после `StandardScaler`:

In [ ]:
from sklearn.preprocessing import StandardScaler

# Два признака с очень разным масштабом
rng3 = np.random.default_rng(3)
n3 = 300
age3  = rng3.normal(40, 10, n3)       # ~40 ± 10
inc3  = rng3.normal(80_000, 20_000, n3) # ~80k ± 20k
X3 = np.column_stack([age3, inc3])

scaler3 = StandardScaler()
X3_scaled = scaler3.fit_transform(X3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for ax, data, title in zip(
    axes,
    [X3, X3_scaled],
    ['До нормировки (исходный масштаб)', 'После StandardScaler'],
):
    ax.boxplot([data[:, 0], data[:, 1]], labels=['age', 'income'], patch_artist=True)
    ax.set_title(title)
    ax.set_ylabel('значение')

fig.suptitle('Распределение признаков до и после масштабирования', fontsize=12)


На исходных данных `age` и `income` несравнимы: у `income` значения в тысячи раз больше. После `StandardScaler` оба признака приведены к одному масштабу.

Проверим, как это влияет на `KNeighborsRegressor`:

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

rng4 = np.random.default_rng(4)
y3 = 2 * age3 + 0.0001 * inc3 + rng4.normal(0, 3, n3)  # зависит от обоих

knn = KNeighborsRegressor(n_neighbors=10)
dt3 = DecisionTreeRegressor(max_depth=5, random_state=0)

results = {}
for name, model in [('KNN', knn), ('DecisionTree', dt3)]:
    score_raw    = cross_val_score(model, X3, y3, cv=5, scoring='r2').mean()
    score_scaled = cross_val_score(
        make_pipeline(StandardScaler(), model), X3, y3, cv=5, scoring='r2'
    ).mean()
    results[name] = {'без нормировки': round(score_raw, 3),
                     'со StandardScaler': round(score_scaled, 3)}

pd.DataFrame(results).T


| Модель | Без нормировки | Со StandardScaler |
|--------|---------------|-------------------|
| **KNN** | Плохо — `income` «перевешивает» расстояние | Хорошо |
| **DecisionTree** | ≈ одинаково | ≈ одинаково |

> **Правило:** `StandardScaler` (или другая нормировка) обязателен для моделей, основанных на расстояниях (`KNN`, `SVM`, `Ridge`) или на градиентах (`LinearRegression`). Деревьям и бустингу — всё равно.

## 2.4 Экстраполяция: что происходит за пределами тренировочных данных

Обучим модели на $x \in [0, 2\pi]$, а затем попросим их предсказать значения за пределами этого диапазона — на $x \in [-\pi, 3\pi]$. Истинная функция — $\sin(x)$, которая прекрасно определена везде.

Как поведут себя наши модели?

In [ ]:
# Обучаем на [0, 2π] — уже обучены выше в models
X_extrap = np.linspace(-np.pi, 3 * np.pi, 600).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

ax.scatter(X_sin, y_sin, s=20, alpha=0.4, color='black', zorder=5, label='train')
ax.plot(X_extrap, np.sin(X_extrap), color='gray', lw=1.5,
        linestyle='--', label='истина sin(x)')

for (name, model), color in zip(models.items(), colors):
    ax.plot(X_extrap, model.predict(X_extrap), lw=2, color=color, label=name)

# Отмечаем границы train
ax.axvline(0,            color='black', lw=0.8, linestyle=':')
ax.axvline(2 * np.pi,    color='black', lw=0.8, linestyle=':')
ax.text(0.05, -1.3, 'train начало', fontsize=8)
ax.text(2 * np.pi - 2.0, -1.3, 'train конец', fontsize=8)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Предсказания моделей за пределами тренировочного диапазона')
ax.legend(ncols=2, fontsize=9)


**Что мы видим:**

| Модель | Поведение за пределами train |
|--------|------------------------------|
| **LinearRegression** | Экстраполирует прямой линией — уходит в бесконечность |
| **DecisionTree** | «Застревает» на граничном значении — константное предсказание |
| **RandomForest** | Аналогично дереву — константа на границах |
| **GradientBoosting** | Аналогично — константа (чуть более гладкая) |
| **KNN** | Возвращает среднее ближайших соседей из train — тоже константа у границ |

Ни одна из моделей не «знает», что истинная функция — это синус. Экстраполяции научиться фактически невозможно без знания о природе данных.

> **Практический вывод:** будьте осторожны, если признаки тестовых данных выходят за диапазоны тренировочных. Деревья вернут «замороженное» граничное значение. Линейные модели продолжат тренд, что иногда хуже.

### Пример с сильным смещением диапазона

Здесь train и test вообще не пересекаются — это классический **OOD (out-of-distribution)** сценарий.

In [ ]:
rng_ood = np.random.default_rng(99)
n_ood = 60

X_ood_train = rng_ood.uniform(0, 5, n_ood).reshape(-1, 1)
y_ood_train = 3 * X_ood_train.ravel() + rng_ood.normal(0, 1, n_ood)

X_ood_test  = np.linspace(0, 15, 300).reshape(-1, 1)

ood_models = {
    'LinearRegression':  LinearRegression(),
    'DecisionTree':       DecisionTreeRegressor(max_depth=4, random_state=0),
    'KNN (k=5)':          KNeighborsRegressor(n_neighbors=5),
}

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.scatter(X_ood_train, y_ood_train, s=20, color='black', alpha=0.5, label='train')
ax.plot(X_ood_test, 3 * X_ood_test, color='gray', lw=1.5, linestyle='--', label='истина y=3x')

for (name, model), color in zip(ood_models.items(), ['tab:blue', 'tab:orange', 'tab:purple']):
    model.fit(X_ood_train, y_ood_train)
    ax.plot(X_ood_test, model.predict(X_ood_test), lw=2, color=color, label=name)

ax.axvspan(5, 15, alpha=0.07, color='red')
ax.text(5.2, 0.5, 'OOD зона (нет в train)', fontsize=9, color='red')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('OOD: train на [0, 5], предсказание на [0, 15]')
ax.legend(fontsize=9)


## 2.5 Преобразование таргета

До сих пор мы обсуждали преобразования признаков. Но иногда полезно преобразовать и **сам таргет** — то, что мы предсказываем.

### Когда это нужно?

Когда таргет имеет **сильно скошенное** (skewed) распределение: например, зарплаты, цены на жильё, количество просмотров. Большинство значений маленькие, но есть единичные гигантские выбросы.

В таких ситуациях:
- Среднеквадратичная ошибка (MSE) штрафует непропорционально сильно за выбросы.
- Линейная модель «тянется» к выбросам и хуже предсказывает типичные значения.

Решение: обучить модель не на $y$, а на $\log(y)$ — это выровнивает распределение. При предсказании результат нужно обратно перевести через $\exp$.

In [ ]:
rng5 = np.random.default_rng(5)
n5 = 400

# Скошенный таргет: log-normal распределение (доходы, цены)
X5 = rng5.normal(0, 1, (n5, 3))
y5_log = 0.5 * X5[:, 0] + 0.3 * X5[:, 1] + rng5.normal(0, 0.4, n5)
y5 = np.exp(y5_log)  # истинный таргет — log-normal

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].hist(y5, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Исходный таргет y (log-normal)')
axes[0].set_xlabel('y')
axes[0].set_ylabel('частота')

axes[1].hist(np.log(y5), bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('После log-преобразования: log(y)')
axes[1].set_xlabel('log(y)')
axes[1].set_ylabel('частота')

fig.suptitle('Распределение таргета до и после log', fontsize=12)


In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor

X5_train, X5_test, y5_train, y5_test = train_test_split(
    X5, y5, test_size=0.25, random_state=0
)

# Вариант 1: Ridge на исходном таргете
model_raw = Ridge()
model_raw.fit(X5_train, y5_train)
mae_raw = mean_absolute_error(y5_test, model_raw.predict(X5_test))

# Вариант 2: вручную log/exp
model_log = Ridge()
model_log.fit(X5_train, np.log(y5_train))
mae_log_manual = mean_absolute_error(y5_test, np.exp(model_log.predict(X5_test)))

# Вариант 3: TransformedTargetRegressor — делает это автоматически
ttr = TransformedTargetRegressor(
    regressor=Ridge(),
    func=np.log,
    inverse_func=np.exp,
)
ttr.fit(X5_train, y5_train)
mae_ttr = mean_absolute_error(y5_test, ttr.predict(X5_test))

print(f'MAE без преобразования таргета:     {mae_raw:.3f}')
print(f'MAE с log(y) вручную:               {mae_log_manual:.3f}')
print(f'MAE с TransformedTargetRegressor:   {mae_ttr:.3f}')


`TransformedTargetRegressor` — обёртка, которая:
1. Применяет `func` к таргету перед обучением.
2. Применяет `inverse_func` к предсказанию перед возвратом.

И самое главное — **вписывается в Pipeline** как обычный estimator:

In [ ]:
# TransformedTargetRegressor в составе полного Pipeline
full_pipe_ttr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', TransformedTargetRegressor(
        regressor=Ridge(),
        func=np.log,
        inverse_func=np.exp,
    )),
])

full_pipe_ttr  # диаграмма


In [ ]:
from sklearn.model_selection import cross_val_score

scores_ttr = cross_val_score(
    full_pipe_ttr, X5, y5,
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)

print(f'CV MAE (Pipeline + TTR): {-scores_ttr.mean():.3f} ± {scores_ttr.std():.3f}')


> **Когда применять log-преобразование таргета?**
> - Таргет строго положительный (цены, выручка, количество)
> - Распределение сильно right-skewed (длинный правый хвост)
> - Остатки модели гетероскедастичны (разброс ошибок растёт с ростом y)
>
> Если таргет уже нормально распределён — log-преобразование не поможет, а может навредить.

---
## Итог

**Pipeline:**
- Устраняет утечку данных — трансформеры обучаются только на train-фолде
- `ColumnTransformer` позволяет применять разные шаги к разным типам признаков
- С `GridSearchCV` можно искать гиперпараметры любого шага через `step__param`

**Особенности моделей:**

| | LinearRegression | DecisionTree / Forest / Boosting | KNN |
|--|------------------|----------------------------------|-----|
| Линейная зависимость | ✅ отлично | ⚠️ ступеньки | ⚠️ приблизительно |
| Нелинейная зависимость | ❌ плохо | ✅ хорошо | ✅ хорошо |
| Экстраполяция | ⚠️ линейный тренд | ❌ константа на границе | ❌ константа на границе |
| Нужна нормировка | ✅ да | ❌ нет | ✅ да (критично!) |

**Преобразование таргета:**
- `TransformedTargetRegressor` — обёртка над любой моделью
- Помогает при log-normal таргетах (цены, выручка)
- Встраивается в Pipeline без дополнительного кода
